# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

**This notebook is the completed solution.** It asks both GPT-4o-mini (streaming) and local Llama 3.2 to explain a short Python snippet.

In [ ]:
# imports

import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [ ]:
# set up environment

for folder in [Path.cwd(), *Path.cwd().parents]:
    env_file = folder / '.env'
    if env_file.exists():
        load_dotenv(env_file, override=True)
        break

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError(
        'OPENAI_API_KEY was not found. Copy .env.example to .env in the repo root and add your key.'
    )

openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

system_prompt = """
You are an expert software engineering tutor.
Explain technical questions clearly and precisely.
Use short examples when they help, and call out common pitfalls.
Respond in markdown. Do not wrap the entire reply in a single fenced code block.
""".strip()


def messages_for(question: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming

display(Markdown(f"### GPT-4o-mini\n\n**Question**\n\n{question}"))

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages_for(question),
    stream=True,
)

gpt_answer = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    gpt_answer += chunk.choices[0].delta.content or ""
    update_display(Markdown(gpt_answer), display_id=display_handle.display_id)

In [ ]:
# Get Llama 3.2 to answer
# Requires: ollama serve  and  ollama pull llama3.2

display(Markdown(f"### Llama 3.2 (Ollama)\n\n**Question**\n\n{question}"))

llama_response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages_for(question),
)
llama_answer = llama_response.choices[0].message.content
display(Markdown(llama_answer))